In [1]:
import pandas as pd
counsel_df = pd.read_csv(r"C:\Users\Pravart singh\Desktop\FY_Project_ALL\Final_Year_Project\datasets\Counseling Conversations.csv")

counsel_df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [2]:
import joblib

tfidf = joblib.load("../models/tfidf_vectorizer.pkl")

main_model = joblib.load("../models/main_emotion_model_tfidf.pkl")
main_encoder = joblib.load("../models/main_emotion_encoder.pkl")

sub_model = joblib.load("../models/sub_emotion_model_tfidf.pkl")
sub_encoder = joblib.load("../models/sub_emotion_encoder.pkl")

In [3]:
X = tfidf.transform(counsel_df["Context"])

main_pred = main_model.predict(X)
sub_pred = sub_model.predict(X)

counsel_df["Main_Emotion"] = main_encoder.inverse_transform(main_pred)
counsel_df["Sub_Emotion"] = sub_encoder.inverse_transform(sub_pred)

In [4]:
counsel_df.head()

,Context,Response,Main_Emotion,Sub_Emotion
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb...",Neutral,neutral
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",Neutral,neutral
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...,Neutral,neutral
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...,Neutral,neutral
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,Neutral,neutral


In [5]:
joblib.dump(
    counsel_df,"../models/counseling_dataset.pkl")

print("Updated counselling dataset saved.")

Updated counselling dataset saved.


In [6]:
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L12-v2"
)

embeddings = embedding_model.encode(
    counsel_df["Context"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype(np.float32)

index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(embeddings)

faiss.write_index(
    index,
    "../models/counseling_faiss.index"
)

print("New FAISS index created successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/110 [00:00<?, ?it/s]

New FAISS index created successfully.


In [21]:
quiz_df = pd.read_csv(r"C:\Users\Pravart singh\Desktop\FY_Project_ALL\Final_Year_Project\datasets\emotion_mcq_dataset.csv")

In [22]:
print(quiz_df.columns.tolist())

['id', 'main_emotion', 'scenario', 'option1', 'option2', 'option3', 'option4', 'correct_answer', 'difficulty', 'Unnamed: 9']


In [23]:
print(quiz_df[["scenario", "Unnamed: 9"]].head())

                                            scenario Unnamed: 9
0  You finally get the job offer you wanted after...        NaN
1  Your best friend throws you a surprise party f...        NaN
2  Finding out your final exams are cancelled thi...        NaN
3  Hitting a new personal record at the gym after...          ,
4  Your crush texts you back immediately with a f...        NaN
